In [50]:
import pandas as pd
import json
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np
from collections import Counter
from scipy.spatial import distance 
from scipy.special import softmax
import torch.nn.functional as F

## Useful Functions

In [13]:
def load_ndjson(data_path) -> list[dict]:

    """
    Loads ndjson files into memory as dictionaries.
    :param data_path: path to ndjson file
    :return: list of dictionaries
    """

    data = []
    with open(data_path, "r") as f:
        for line in f:
            if line.strip():  # avoid empty lines
                data.append(json.loads(line))

    return data

In [ ]:
def get_internal_probs(measurements: list[dict]):
    
    """
    :param measurements: list of Measurement objects (or equivelant dictionaries)
    """
    
    # calculate probs from mean logits (should be all the same, though)
    internal_probs = softmax(np.mean([np.array(l['logits']) for l in measurements], axis=0))
    
    return internal_probs

In [ ]:
def get_generation_probs(measurements: list[dict]):
    
    """
    
    :param measurements: list of Measurement objects (or equivelant dictionaries)
    """
    # pronoun set is determined by the first measurement
    pnouns = measurements[0]['context']['pronouns_2']
    
    # calculate empirical generation probabilities (remove anything not in the list of pronouns)
    generation_counter = Counter([l['measurement']['BLANK'] for l in measurements])
    generation_counter_clean = {k: generation_counter[k] for k in pnouns}
    num_valid_measurements = np.sum(list(generation_counter_clean.values()))
    generation_probs = np.array(list(generation_counter_clean.values()))/num_valid_measurements
    
    return generation_probs

In [54]:
def get_generation_logit_distortion(measurements: list[dict]):
    
    """
    Measure of the difference between the (deterministic) internal beliefs of a model and the (observed) generation
    frequency of those same tokens. 
    
    Distance is measured as L1 (Manhattan) Distance because this a discrete, unordered distribution.
    
    :param measurements: list of Measurement objects (or equivelant dictionaries)
    """
    
    internal_probs = get_internal_probs(measurements)
    generation_probs = get_generation_probs(measurements)
    
    # L1 distance of the two arrays 
    return distance.cityblock(internal_probs, generation_probs)

## Data Loading

In [4]:
DATA_DIR = Path("../../")

In [11]:
LLAMA32_PATH = DATA_DIR / "one_pronoun_measurements_Llama-3.2-1B-Instruct_0.5.ndjson"

In [14]:
llama32 = load_ndjson(LLAMA32_PATH)

In [39]:
llama32_10 = [c for c in llama32 if (c['index']==10)]

In [63]:
llama32_10_na_0_fwd = [c for c in llama32 if (c['index']==10) and 
                 (c['context']['sentence_1'] is None) and 
                 (c['context']['pnoun_order'] == [None, 0]) and
                    (c['context']['sent_order'] == [0, 1])]

In [64]:
llama32_10_na_0_fwd_probs = softmax(np.mean([np.array(l['logits']) for l in llama32_10_na], 
                                      axis=0))
llama32_10_na_0_fwd_probs

array([0.15329549, 0.84670451])

In [65]:
llama32_10_na_0_fwd[0]

{'index': 10,
 'context': {'sent_order': [0, 1],
  'pnoun_order': [None, 0],
  'sentence_1': None,
  'sentence_2': 'The customer asked to speak with the manager because BLANK wanted to fix the billing error quickly.',
  'pronouns_1': ['he', 'she'],
  'pronouns_2': ['he', 'she']},
 'measurement': {'BLANK': 'she'},
 'probabilities': None,
 'logits': [10.015625, 10.0703125]}

In [66]:
Counter([l['measurement']['BLANK'] for l in llama32_10_na_0_fwd])

Counter({'she': 67, 'he': 33})

In [32]:
#logits_llama32_10 = np.mean([np.array(l['logits']) for l in llama32_10], axis=0)
#softmax(logits_llama32_10)

In [67]:
get_generation_logit_distortion(llama32_10_na_0_fwd)

0.31266306276021893

# Belief Contextuality

In [78]:
len([l for l in llama32 if l['context']['sentence_1'] is not None])

0